In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [2]:
import json
from uuid import uuid4
from pathlib import Path
from textwrap import dedent

import pandas as pd

from rich import print
from openai import OpenAI
from pydantic import BaseModel
from openai.lib._pydantic import to_strict_json_schema

In [3]:
client = OpenAI()

In [4]:
SYSTEM_PROMPTS = {
    "beir_corpus": dedent("""
                          Translate this Sundanese text including it's title and body into English.
                          Beware that it might contain a specific Sundanese context or nuances that must be correctly interpreted and not translated literally.
                          """),
    "beir_query": dedent("""
                         Translate this Sundanese with possible Indonesian text into English.
                         Beware that it might contain a specific Sundanese context or nuances that must be correctly interpreted and not translated literally.
                         """),
    "triplet": dedent("""
                      Translate this Sundanese passages into English.
                      You will be provided with a query along with the relevant and irrelevant answers.
                      Beware that it might contain a specific Sundanese context or nuances that must be correctly interpreted and not translated literally.
                      """),
}

## BEIR

### Corpus

In [5]:
df_corpus = pd.read_json("../data/cleaned/corpus.jsonl", lines=True)
df_corpus.head()

,_id,title,text
0,1bcfc529-9788-4f8b-a8e4-c2780922cb9e,Wakil Dubes Walanda Gumbira Ningali Holland In...,bogor hotél institut (bhi) gawé bareng jeung f...
1,7ad64ffc-449a-4218-b3fb-a9ad92925068,Warga Bogor Mapag Taun anyar Islam,bogor - datang ton anyar islam 1431-hijréh pap...
2,6c714fec-c629-4604-bfac-0096e1a0f624,Warugan Lemah: Pola Lembur Urang Sunda Buhun,naskah warugan lemah kandelna ngan tilu lempir...
3,5669e653-63ab-49b4-be39-7bf07f2eefe7,Manfaat Olahraga Pikeun Kasehatan,anu ku urang tos terang olahraga teh penting p...
4,bfc35e44-876f-4ddd-a7c3-dbe6339a27f1,DINA JANDÉLA INDUNG,"méméh layung kubur panineungan dina jandéla, g..."


In [6]:
def corpus_item_text(value):
    return "<title>" + value.title + "</title>\n<body>" + value.text + "</body>"

In [7]:
class BEIRCorpusItem(BaseModel):
    title_english: str
    body_english: str

In [8]:
completion_beir_corpus = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=BEIRCorpusItem,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["beir_corpus"],
        },
        {
            "role": "user",
            "content": corpus_item_text(df_corpus.iloc[0]),
        },
    ],
)

print(completion_beir_corpus)

ParsedChatCompletion[BEIRCorpusItem](
    id='chatcmpl-BVWCA8z94MgNf5fp7PnHd4oK0JLva',
    choices=[
        ParsedChoice[BEIRCorpusItem](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[BEIRCorpusItem](
                content='{"title_english":"Deputy Dutch Ambassador Delighted to See Holland Indonesia 
Festival","body_english":"Bogor Hotel Institute (BHI) collaborated with the Forum Indonesian Netherlands (FINED) 
and Salak the Heritage Hotel for the \\"Holland Indonesia Festival\\" held at Salak Hotel, Jalan Ir Jondang, Bogor 
City, (15/11/2009). The festival, which was opened by the Dutch deputy ambassador to Indonesia, Annemieke Ruigrok, 
aimed to introduce the culture of both countries to the public. The Dutch deputy ambassador, Annemieke Ruigrok, 
expressed her delight at the event. \\"Indonesia and the Netherlands have a historic relationship,\\" she said. 
\\"Although the Netherlands has a past that is not well regarded in the eyes of the Indonesian people, it does not 
erase that relationship,\\" she added. Looking at the historical past, Indonesia and the Netherlands share several 
similarities in terms of culture. Many Dutch citizens have an appreciation for the arts originating from Indonesia.
\\"The Dutch are very fond of Indonesian cuisine,\\" she continued in Dutch. \\"I appreciate the hospitality of the
Indonesian people. I am happy to be in Indonesia with such friendly people,\\" she reiterated. Various cultural 
elements belonging to the Indonesian nation were showcased at this festival, including batik and angklung art. 
Ruigrok also paid attention to batik artisans demonstrating the process of making hand-drawn batik. According to R.
Ay. Suni Wijogawati, representative head of FINED, this cultural festival was only held for a day. Prior to its 
execution in Bogor, the festival had also taken place in other cities. Suni, who is also the management assistant 
at Erasmus Huis, the Dutch cultural center in Jakarta, explained that the series of events will continue to be 
promoted. The sole aim is to strengthen the relationship between Indonesia and the Netherlands."}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=BEIRCorpusItem(
                    title_english='Deputy Dutch Ambassador Delighted to See Holland Indonesia Festival',
                    body_english='Bogor Hotel Institute (BHI) collaborated with the Forum Indonesian Netherlands 
(FINED) and Salak the Heritage Hotel for the "Holland Indonesia Festival" held at Salak Hotel, Jalan Ir Jondang, 
Bogor City, (15/11/2009). The festival, which was opened by the Dutch deputy ambassador to Indonesia, Annemieke 
Ruigrok, aimed to introduce the culture of both countries to the public. The Dutch deputy ambassador, Annemieke 
Ruigrok, expressed her delight at the event. "Indonesia and the Netherlands have a historic relationship," she 
said. "Although the Netherlands has a past that is not well regarded in the eyes of the Indonesian people, it does 
not erase that relationship," she added. Looking at the historical past, Indonesia and the Netherlands share 
several similarities in terms of culture. Many Dutch citizens have an appreciation for the arts originating from 
Indonesia. "The Dutch are very fond of Indonesian cuisine," she continued in Dutch. "I appreciate the hospitality 
of the Indonesian people. I am happy to be in Indonesia with such friendly people," she reiterated. Various 
cultural elements belonging to the Indonesian nation were showcased at this festival, including batik and angklung 
art. Ruigrok also paid attention to batik artisans demonstrating the process of making hand-drawn batik. According 
to R. Ay. Suni Wijogawati, representative head of FINED, this cultural festival was only held for a day. Prior to 
its executio

### Queries

In [9]:
df_queries = pd.read_json("../data/cleaned/queries.jsonl", lines=True)
df_queries.head()

,_id,text
0,99b78da2-54b4-4afa-b4c8-06df0dfb53d8,Holland Indonésé Festival di Bogor
1,42a37e80-e810-4813-a551-09dcbd471fa6,apa tujuan festival budaya di Bogor
2,5a0b9395-c28a-4c33-b5cc-5c502ee78790,siapa wakil duta besar Walanda
3,b821e85d-a6c3-4413-a0ba-d09ab1b72c2a,hubungan antara Indonésé jeung Walanda
4,37959229-85cc-432d-97e4-b437002efb17,acara naon anu dipidangkeun di festival


In [10]:
def beir_query_text(value):
    return "<query>" + value.text + "</query>"

In [11]:
class BEIRQueryItem(BaseModel):
    query_english: str

In [12]:
completion_beir_query = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=BEIRQueryItem,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["beir_query"],
        },
        {
            "role": "user",
            "content": beir_query_text(df_queries.iloc[0]),
        },
    ],
)

print(completion_beir_query)

ParsedChatCompletion[BEIRQueryItem](
    id='chatcmpl-BVWCPI4jvhVAy96MRc8vqSIVrSdes',
    choices=[
        ParsedChoice[BEIRQueryItem](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[BEIRQueryItem](
                content='{"query_english":"Holland Indonesian Festival in Bogor"}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=BEIRQueryItem(query_english='Holland Indonesian Festival in Bogor'),
                annotations=[]
            )
        )
    ],
    created=1746851641,
    model='gpt-4o-mini-2024-07-18',
    object='chat.completion',
    service_tier='default',
    system_fingerprint='fp_dbaca60df0',
    usage=CompletionUsage(
        completion_tokens=14,
        prompt_tokens=105,
        total_tokens=119,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)

## Triplet

In [13]:
df_triplet = pd.read_json("../data/cleaned/triplet.jsonl", lines=True)
df_triplet.head()

,query,positive,negative
0,Naon tujuan Holland Indonésé Festival?,Festival anu buka wakil duta besar walanda keu...,Urang walanda pohara resep ku masak urang indo...
1,Kapan Holland Indonésé Festival dilaksanakeun?,"Festival ieu dilaksanakeun di hotél salak, jal...",Sagalana rupa budaya milik bangsa indonésé dih...
2,Saha nu nyarios ngeunaan hubungan indonésé jeu...,"Wakil duta besar walanda, annemieke ruigrok ka...",Nurutkeun r. ay. suni wijogawati laku wakil pu...
3,Ala naon anu dipidangkeun di festival budaya ieu?,Sagalana rupa budaya milik bangsa indonésé dih...,Kuring gumbira aya di indonésé nu jalma saromé...
4,Saha anu mendakan cara sieun batik tulis?,Ruigrok sempet nengetan pengrajin batik mrakté...,Festival budaya ieu ngan gel sapoé.


In [14]:
def triplet_item_text(value):
    return "<query>" + value.query + "</query>\n<positive>" + value.positive + "</positive>\n<negative>" + value.negative + "</negative>"

In [15]:
class TripletItem(BaseModel):
    query_english: str
    positive_english: str
    negative_english: str

In [16]:
completion_triplet = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=TripletItem,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["beir_query"],
        },
        {
            "role": "user",
            "content": triplet_item_text(df_triplet.iloc[0]),
        },
    ],
)

print(completion_triplet)

ParsedChatCompletion[TripletItem](
    id='chatcmpl-BVWCbHmXiKNQarydkDz7lcZkAXZIE',
    choices=[
        ParsedChoice[TripletItem](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[TripletItem](
                content='{"query_english":"What is the purpose of the Holland Indonesian 
Festival?","positive_english":"The festival, which is attended by the Dutch ambassador to Indonesia, Annemieke 
Ruigrok, aims to introduce the culture of these two countries to the public.","negative_english":"The Dutch really 
enjoy Indonesian cuisine, then in the Dutch language."}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=TripletItem(
                    query_english='What is the purpose of the Holland Indonesian Festival?',
                    positive_english='The festival, which is attended by the Dutch ambassador to Indonesia, 
Annemieke Ruigrok, aims to introduce the culture of these two countries to the public.',
                    negative_english='The Dutch really enjoy Indonesian cuisine, then in the Dutch language.'
                ),
                annotations=[]
            )
        )
    ],
    created=1746851653,
    model='gpt-4o-mini-2024-07-18',
    object='chat.completion',
    service_tier='default',
    system_fingerprint='fp_dbaca60df0',
    usage=CompletionUsage(
        completion_tokens=72,
        prompt_tokens=205,
        total_tokens=277,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)

## TODO: Generate OpenAI Batch Request

### Batch Request Generator

In [17]:
def generate_batch(df: pd.DataFrame, system_prompt: str, base_model: BaseModel, formatter_fun):
    for row in df.itertuples():
        custom_id = str(uuid4())
        job_data = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-4o-mini",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": formatter_fun(row)},
                ],
                "response_format": {
                    "type": "json_schema",
                    "json_schema": {
                        "name": base_model.__name__,
                        "strict": True,
                        "schema": to_strict_json_schema(base_model),
                    },
                },
            },
        }
        
        yield (custom_id, str(row.Index), job_data)

In [18]:
# item = next(generate_batch(df_corpus, SYSTEM_PROMPTS["beir_corpus"], BEIRCorpusItem, corpus_item_text))
# item = next(generate_batch(df_queries, SYSTEM_PROMPTS["beir_query"], BEIRQueryItem, beir_query_text))
item = next(generate_batch(df_triplet, SYSTEM_PROMPTS["triplet"], TripletItem, triplet_item_text))

print(item)

(
    '18d37a46-5d16-4d75-9cbe-9b67e9eec8fa',
    '0',
    {
        'custom_id': '18d37a46-5d16-4d75-9cbe-9b67e9eec8fa',
        'method': 'POST',
        'url': '/v1/chat/completions',
        'body': {
            'model': 'gpt-4o-mini',
            'messages': [
                {
                    'role': 'system',
                    'content': '\nTranslate this Sundanese passages into English.\nYou will be provided with a 
query along with the relevant and irrelevant answers.\nBeware that it might contain a specific Sundanese context or
nuances that must be correctly interpreted and not translated literally.\n'
                },
                {
                    'role': 'user',
                    'content': '<query>Naon tujuan Holland Indonésé Festival?</query>\n<positive>Festival anu buka 
wakil duta besar walanda keur indonésé, annemieke ruigrok boga tuju pikeun ngawanohkeun budaya anu dipiboga ku do 
nagara ieu ka masarakat.</positive>\n<negative>Urang walanda pohara resep ku masak urang indonésé, terus dina basa 
walanda.</negative>'
                }
            ],
            'response_format': {
                'type': 'json_schema',
                'json_schema': {
                    'name': 'TripletItem',
                    'strict': True,
                    'schema': {
                        'properties': {
                            'query_english': {'title': 'Query English', 'type': 'string'},
                            'positive_english': {'title': 'Positive English', 'type': 'string'},
                            'negative_english': {'title': 'Negative English', 'type': 'string'}
                        },
                        'required': ['query_english', 'positive_english', 'negative_english'],
                        'title': 'TripletItem',
                        'type': 'object',
                        'additionalProperties': False
                    }
                }
            }
        }
    }
)

In [19]:
def persist_batch(df: pd.DataFrame, schema: BaseModel, format_fun, kind: str):
    batch_req_path = Path(f"../data/llm-gen/translated/{kind}_batch.jsonl")
    batch_map_path = Path(f"../data/llm-gen/translated/{kind}_map.jsonl")
    batch_map_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(batch_req_path, "w") as fm, open(batch_map_path, "w") as mm:
        batch_iter = generate_batch(df, SYSTEM_PROMPTS[kind], schema, format_fun)
        for custom_id, doc_id, req in batch_iter:
            json.dump(req, fm)
            fm.write("\n")

            json.dump({"custom_id": custom_id, "doc_index": doc_id}, mm)
            mm.write("\n")
    
    return batch_req_path, batch_map_path

In [20]:
beir_corpus_req_path, beir_corpus_map_path = persist_batch(df_corpus, BEIRCorpusItem, corpus_item_text, "beir_corpus")
beir_query_req_path, beir_query_map_path = persist_batch(df_queries, BEIRQueryItem, beir_query_text, "beir_query")
triplet_req_path, triplet_map_path = persist_batch(df_triplet, TripletItem, triplet_item_text, "triplet")

### Submit Batch Requests

In [21]:
def submit_batch(path):
    batch_file = client.files.create(file=open(path, "rb"), purpose="batch")

    return client.batches.create(
        input_file_id=batch_file.id, 
        endpoint="/v1/chat/completions", 
        completion_window="24h"
    )

In [22]:
triplet_batch = submit_batch(triplet_req_path.resolve())
print(triplet_batch)

Batch(
    id='batch_681ed8f1a9d481909d0c19b6f0a8d969',
    completion_window='24h',
    created_at=1746852081,
    endpoint='/v1/chat/completions',
    input_file_id='file-Lfz56i7emPQTgaebnsT91N',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746938481,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)

In [23]:
beir_query_batch = submit_batch(beir_query_req_path.resolve())
print(beir_query_batch)

Batch(
    id='batch_681ed92d1b6881909495bbe442fba25b',
    completion_window='24h',
    created_at=1746852141,
    endpoint='/v1/chat/completions',
    input_file_id='file-RLAmgsk9bHmGv1uGHYSux5',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746938541,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)

In [24]:
beir_corpus_batch = submit_batch(beir_corpus_req_path.resolve())
print(beir_corpus_batch)

Batch(
    id='batch_68180c78133881908cb8388e303a966e',
    completion_window='24h',
    created_at=1746406520,
    endpoint='/v1/chat/completions',
    input_file_id='file-G1fyzVU2mi2NXceLGPyQbQ',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746492920,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)